# Experiment Theory Notebook (Evaluation Notes)

## Scope of this cookbook

This cookbook proves the vanishing-gradient problem and the role of residual identity mapping on SVHN through three executable experiments:

1. Simple CNN baseline (no residual blocks, no skip)
2. ResNet-style network without skip addition
3. ResNet-style network with skip addition

## Core concepts you must be able to explain

### 1) Vanishing gradients

For deep feedforward stacks, gradients backpropagate as chained products:

$$\frac{\partial \mathcal{L}}{\partial W_l} = \frac{\partial \mathcal{L}}{\partial a_L} \prod_{i=l}^{L-1}\frac{\partial a_{i+1}}{\partial a_i}$$

Repeated multiplication by factors with magnitude often below 1 can shrink early-layer gradients and updates.

### 2) Residual formulation

Residual blocks re-parameterize mapping as:

$$y = F(x) + x$$

with identity shortcut $x$ (or projection shortcut when dimensions change).

### 3) Why skip helps backprop

$$\frac{\partial y}{\partial x} = \frac{\partial F(x)}{\partial x} + I$$

Identity provides a direct route for gradient flow, reducing collapse risk in earlier layers.

## What is measured per convolution layer

- Gradient norm (L2), batch-wise and epoch-wise
- Weight norm (L2), epoch-wise
- Weight delta (L2), epoch-wise

These directly measure whether each layer is still trainable via backpropagation.

## Judgment checklist

- Can you show weaker early-layer update behavior in no-skip variants?
- Can you show improved update flow when skip addition is enabled?
- Can you explain the math-to-observation link using layer-wise logs and plots?

In [ ]:
# Notebook execution order for full evaluation
full_notebook_order = [
    '10_simple_cnn_full.ipynb',
    '20_resnet_no_skip_full.ipynb',
    '30_resnet_with_skip_full.ipynb',
    '40_tables_and_architecture.ipynb',
    '50_compare_all_full.ipynb',
]
for i, nb in enumerate(full_notebook_order, start=1):
    print(f'{i}. {nb}')

In [ ]:
# Standard hyperparameter policy used across notebooks
experiment_config = {
    'dataset': 'SVHN',
    'batch_size': 128,
    'epochs': 20,
    'optimizer': 'Adam',
    'learning_rate': 1e-3,
    'scheduler': 'StepLR(step=10, gamma=0.5)',
    'loss': 'CrossEntropyLoss',
    'init': 'Kaiming (He) Normal',
}
for k, v in experiment_config.items():
    print(f'{k}: {v}')

## Oral explanation template (for viva/judging)

Use this compact sequence while presenting:

1. Start from vanishing-gradient chain-rule math
2. Define difference between plain mapping $y=F(x)$ and residual mapping $y=F(x)+x$
3. Explain why identity term adds gradient highway
4. Show experiment outputs layer-wise (not just final accuracy)
5. Conclude using first-layer vs last-layer gradient and delta evidence

The strongest argument is **per-layer training dynamics**, not only top-line accuracy.